In [1]:
import os
import json
from pathlib import Path
import pandas as pd

os.chdir(r'C:\Users\HP\Documents\Economy\economia_c')
print("Directorio:", Path.cwd())

Directorio: C:\Users\HP\Documents\Economy\economia_c


In [2]:
RUTA = Path.cwd() / "data" / "raw" / "SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx"

df = pd.read_excel(RUTA, sheet_name=0, header=None)

print(f"Shape: {df.shape}")
print()
print("Primeras 3 filas x 8 columnas:")
print(df.iloc[:3, :8].to_string())

Shape: (28, 211)

Primeras 3 filas x 8 columnas:
                                  0                    1                    2                    3                    4                    5                    6                    7
0  -en miles de puestos de trabajo-  2009-01-01 00:00:00  2009-02-01 00:00:00  2009-03-01 00:00:00  2009-04-01 00:00:00  2009-05-01 00:00:00  2009-06-01 00:00:00  2009-07-01 00:00:00
1                            Bs. As              1787.64             1769.223             1745.913             1729.917             1723.218             1724.903             1722.949
2                              CABA             1469.941             1467.029             1466.167             1454.724             1451.374             1450.695             1449.348


In [3]:
# Fila 0: fechas
fechas = pd.to_datetime(df.iloc[0, 1:], errors="coerce")

# Fila 12: La Rioja (en miles de puestos)
fila_rioja = df.iloc[12, 1:].astype(float)

# Armar DataFrame
serie = pd.DataFrame({
    "fecha": fechas.values,
    "empleo_miles": fila_rioja.values,
}).dropna()

# Filtrar desde 2022
serie = serie[serie["fecha"] >= "2022-01-01"].copy()

# Convertir a puestos
serie["empleo"] = (serie["empleo_miles"] * 1000).round(0).astype(int)

# Calcular variación interanual
serie["var_ia"] = (serie["empleo"].pct_change(12) * 100).round(2)

# Etiquetas formato "Ene-22"
meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
serie["label"] = serie["fecha"].apply(lambda d: f"{meses[d.month-1]}-{str(d.year)[2:]}")

print(f"Período: {serie['label'].iloc[0]} → {serie['label'].iloc[-1]}")
print(f"Meses: {len(serie)}")

Período: Abr-23 → Jun-26
Meses: 39


In [4]:
print(f"{'Mes':<10} {'Empleo':>10} {'Var i.a.':>12}")
print("-" * 35)
for _, row in serie.iterrows():
    var = f"{row['var_ia']:+.2f}%" if pd.notna(row['var_ia']) else "—"
    print(f"{row['label']:<10} {row['empleo']:>10,} {var:>12}".replace(",", "."))
    

Mes            Empleo     Var i.a.
-----------------------------------
Abr-23         34.866            —
May-23         34.329            —
Jun-23         33.740            —
Jul-23         32.593            —
Ago-23         31.886            —
Sep-23         31.274            —
Oct-23         31.051            —
Nov-23         30.693            —
Dic-23         29.977            —
Ene-24         31.289            —
Feb-24         36.709            —
Mar-24         33.582            —
Abr-24         31.589       -9.40%
May-24         30.678      -10.64%
Jun-24         28.830      -14.55%
Jul-24         28.005      -14.08%
Ago-24         27.917      -12.45%
Sep-24         28.020      -10.40%
Oct-24         27.873      -10.23%
Nov-24         27.826       -9.34%
Dic-24         27.519       -8.20%
Ene-25         28.155      -10.02%
Feb-25         33.550       -8.61%
Mar-25         32.495       -3.24%
Abr-25         31.597       +0.03%
May-25         30.835       +0.51%
Jun-25         28.9

In [5]:
max_row = serie.loc[serie["empleo"].idxmax()]
min_row = serie.loc[serie["empleo"].idxmin()]
ult_row = serie.iloc[-1]

print("=" * 60)
print("ESTADÍSTICAS CLAVE — EMPLEO PRIVADO FORMAL LA RIOJA")
print("=" * 60)
print(f"  Pico:         {max_row['empleo']:,} ({max_row['label']})".replace(",", "."))
print(f"  Mínimo:       {min_row['empleo']:,} ({min_row['label']})".replace(",", "."))
print(f"  Último dato:  {ult_row['empleo']:,} ({ult_row['label']})".replace(",", "."))
if pd.notna(ult_row["var_ia"]):
    print(f"  Var i.a.:     {ult_row['var_ia']:+.2f}%")

ESTADÍSTICAS CLAVE — EMPLEO PRIVADO FORMAL LA RIOJA
  Pico:         36.709 (Feb-24)
  Mínimo:       26.385 (Dic-25)
  Último dato:  28.656 (Jun-26)
  Var i.a.:     -0.84%


In [6]:
labels = serie["label"].tolist()
empleo = serie["empleo"].tolist()
var_ia = [None if pd.isna(v) else v for v in serie["var_ia"].tolist()]

print("=" * 70)
print("COPIÁ ESTOS 3 BLOQUES Y PEGALOS EN EL HTML")
print("=" * 70)
print()
print("// LABELS:")
print(json.dumps(labels, ensure_ascii=False))
print()
print("// EMPLEO:")
print(json.dumps(empleo))
print()
print("// VAR IA:")
print(json.dumps(var_ia))

COPIÁ ESTOS 3 BLOQUES Y PEGALOS EN EL HTML

// LABELS:
["Abr-23", "May-23", "Jun-23", "Jul-23", "Ago-23", "Sep-23", "Oct-23", "Nov-23", "Dic-23", "Ene-24", "Feb-24", "Mar-24", "Abr-24", "May-24", "Jun-24", "Jul-24", "Ago-24", "Sep-24", "Oct-24", "Nov-24", "Dic-24", "Ene-25", "Feb-25", "Mar-25", "Abr-25", "May-25", "Jun-25", "Jul-25", "Ago-25", "Sep-25", "Oct-25", "Nov-25", "Dic-25", "Ene-26", "Feb-26", "Mar-26", "Abr-26", "May-26", "Jun-26"]

// EMPLEO:
[34866, 34329, 33740, 32593, 31886, 31274, 31051, 30693, 29977, 31289, 36709, 33582, 31589, 30678, 28830, 28005, 27917, 28020, 27873, 27826, 27519, 28155, 33550, 32495, 31597, 30835, 28900, 28103, 27404, 27110, 26904, 26430, 26385, 27634, 34539, 33471, 31358, 30305, 28656]

// VAR IA:
[null, null, null, null, null, null, null, null, null, null, null, null, -9.4, -10.64, -14.55, -14.08, -12.45, -10.4, -10.23, -9.34, -8.2, -10.02, -8.61, -3.24, 0.03, 0.51, 0.24, 0.35, -1.84, -3.25, -3.48, -5.02, -4.12, -1.85, 2.95, 3.0, -0.76, -1.72, -0.8

In [8]:
import pandas as pd
from pathlib import Path

RUTA_ANUAL = Path.cwd() / "data" / "raw" / "provinciales_serie_empleo_anual_3.xlsx"

xls = pd.ExcelFile(RUTA_ANUAL)
print("Hojas del archivo anual:")
print(xls.sheet_names)

Hojas del archivo anual:
['Carátula', 'Metodología', 'Indice', 'T1', 'T2', 'T3a', 'T3b', 'T3c', 'T4a', 'T4b', 'T5']


In [9]:
import pandas as pd
from pathlib import Path

RUTA_ANUAL = Path.cwd() / "data" / "raw" / "provinciales_serie_empleo_anual_3.xlsx"

# Leer la hoja "Indice" para ver qué contiene cada tabla
df_indice = pd.read_excel(RUTA_ANUAL, sheet_name="Indice", header=None)
print("=== HOJA INDICE ===")
print(df_indice.to_string())

=== HOJA INDICE ===
     0                                                                                                                                                                       1
0  NaN                                                                                                                                                                  Índice
1  NaN                                                                                                                                                                     NaN
2  NaN                            Cuadro 1.  Puestos de trabajo asalariado registrado del sector privado según grandes sectores de actividad económica  -IV° trimestre de 2025
3  NaN                                                                                                                                                                     NaN
4  NaN  Cuadro 2. Puestos de trabajo asalariado registrado del sector privado según grandes sectores de a

In [10]:
import pandas as pd
from pathlib import Path

RUTA_ANUAL = Path.cwd() / "data" / "raw" / "provinciales_serie_empleo_anual_3.xlsx"

df_t5 = pd.read_excel(RUTA_ANUAL, sheet_name="T5", header=None)
print(f"Shape: {df_t5.shape}")
print()
print("Primeras 15 filas x 8 columnas:")
print(df_t5.iloc[:15, :8].to_string())

Shape: (205, 31)

Primeras 15 filas x 8 columnas:
                                                                                      0           1           2           3           4                 5           6           7
0   Cuadro 5a. Puestos de trabajo por sector - promedios anuales- en miles, 1996-2025.          NaN         NaN         NaN         NaN  Volver al índice         NaN         NaN
1                                                    Provincia en la que declara empleo  1996.00000  1997.00000  1998.00000  1999.00000              2000  2001.00000  2002.00000
2                                                                       Partidos de GBA   644.13225   688.06925   760.12050   744.08150         721.94575   712.03750   640.58850
3                                                        Agricultura, ganadería y pesca     4.48500     4.80375     5.09850     5.50900            3.9855     3.96950     3.91100
4                                                           

In [11]:
import pandas as pd
from pathlib import Path

RUTA_ANUAL = Path.cwd() / "data" / "raw" / "provinciales_serie_empleo_anual_3.xlsx"
df_t5 = pd.read_excel(RUTA_ANUAL, sheet_name="T5", header=None)

# Buscar la fila donde aparece "La Rioja" en columna 0
for i in range(len(df_t5)):
    val = str(df_t5.iloc[i, 0])
    if "Rioja" in val and "La Rioja" in val:
        print(f"Fila {i}: {val}")
        # Mostrar las siguientes 7 filas (los 7 sectores)
        for j in range(i, min(i+8, len(df_t5))):
            print(f"  {j}: {df_t5.iloc[j, 0]}")
        break

Fila 98: La Rioja
  98: La Rioja
  99: Agricultura, ganadería y pesca
  100: Minería y petróleo
  101: Industria
  102: Comercio
  103: Servicios
  104: Electricidad, gas y agua
  105: Construcción


In [12]:
import pandas as pd
from pathlib import Path

RUTA_ANUAL = Path.cwd() / "data" / "raw" / "provinciales_serie_empleo_anual_3.xlsx"
df_t5 = pd.read_excel(RUTA_ANUAL, sheet_name="T5", header=None)

# Fila 1 tiene los años
anios = df_t5.iloc[1, :].tolist()

# Buscar columna de 2025
col_2025 = None
for i, a in enumerate(anios):
    try:
        if int(float(a)) == 2025:
            col_2025 = i
            break
    except (ValueError, TypeError):
        continue

print(f"Columna de 2025: {col_2025}")
print()

# Extraer sectores de La Rioja (filas 99-105)
sectores = []
for i in range(99, 106):
    nombre = str(df_t5.iloc[i, 0])
    valor_miles = df_t5.iloc[i, col_2025]
    if pd.notna(valor_miles):
        valor_puestos = int(float(valor_miles) * 1000)
        sectores.append((nombre, valor_puestos))
        print(f"  {nombre:<40} {valor_puestos:>10,} puestos".replace(",", "."))

print()
print("=" * 60)
print("PARA PEGAR EN EL HTML:")
print("=" * 60)
print()
print("// SECTORES:")
print([s[0] for s in sectores])
print()
print("// VALORES:")
print([s[1] for s in sectores])

Columna de 2025: 30

  Agricultura. ganadería y pesca                4.945 puestos
  Minería y petróleo                              126 puestos
  Industria                                     8.546 puestos
  Comercio                                      6.720 puestos
  Servicios                                     8.728 puestos
  Electricidad. gas y agua                        852 puestos
  Construcción                                  1.147 puestos

PARA PEGAR EN EL HTML:

// SECTORES:
['Agricultura, ganadería y pesca', 'Minería y petróleo', 'Industria', 'Comercio', 'Servicios', 'Electricidad, gas y agua', 'Construcción']

// VALORES:
[4945, 126, 8546, 6720, 8728, 852, 1147]


In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

os.chdir(r'C:\Users\HP\Documents\Economy\economia_c')
print("Directorio:", Path.cwd())

Directorio: C:\Users\HP\Documents\Economy\economia_c


In [4]:
RUTA = Path.cwd() / "data" / "raw" / "SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(RUTA, sheet_name=0, header=None)

fechas = pd.to_datetime(df.iloc[0, 1:], errors="coerce")
fila_rioja = df.iloc[12, 1:].astype(float)

serie = pd.DataFrame({
    "fecha": fechas.values,
    "empleo_miles": fila_rioja.values,
}).dropna()

serie = serie[serie["fecha"] >= "2022-01-01"].copy()
serie["empleo"] = (serie["empleo_miles"] * 1000).round(0).astype(int)
serie["var_ia"] = (serie["empleo"].pct_change(12) * 100).round(2)

meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
serie["label"] = serie["fecha"].apply(lambda d: f"{meses[d.month-1]}-{str(d.year)[2:]}")

print(f"Período: {serie['label'].iloc[0]} → {serie['label'].iloc[-1]}")
print(f"Meses: {len(serie)}")

Período: Abr-23 → Jun-26
Meses: 39


In [5]:
# Preparar serie temporal
serie_ts = serie.copy()
serie_ts = serie_ts.set_index("fecha").sort_index()
serie_ts = serie_ts.asfreq("MS")

# Paso 1: tendencia por promedio móvil centrado de 12 meses
serie_ts["tendencia"] = serie_ts["empleo"].rolling(window=12, center=True).mean()

# Paso 2: componente estacional = original / tendencia
serie_ts["ratio"] = serie_ts["empleo"] / serie_ts["tendencia"]

# Paso 3: factor estacional por mes
serie_ts["mes"] = serie_ts.index.month
factores = serie_ts.groupby("mes")["ratio"].mean().to_dict()

# Paso 4: desestacionalizar
serie_ts["factor"] = serie_ts["mes"].map(factores)
serie_ts["empleo_desest"] = serie_ts["empleo"] / serie_ts["factor"]

# Volver a formato original
serie_desest = serie_ts.reset_index()

# Mostrar comparación
print(f"{'Mes':<10} {'Original':>10} {'Desest.':>10} {'Factor':>10}")
print("-" * 45)
for _, row in serie_desest.iterrows():
    mes = row["fecha"].strftime("%b-%y")
    orig = int(row["empleo"])
    des = int(round(row["empleo_desest"]))
    fac = row["factor"]
    print(f"{mes:<10} {orig:>10,} {des:>10,} {fac:>10.3f}".replace(",", "."))

# Estadísticas
print()
print("=" * 60)
print("ESTADÍSTICAS DESESTACIONALIZADAS")
print("=" * 60)
max_idx = serie_desest["empleo_desest"].idxmax()
min_idx = serie_desest["empleo_desest"].idxmin()
ult_idx = serie_desest.index[-1]

print(f"  Pico:    {int(serie_desest.loc[max_idx, 'empleo_desest']):,} ({serie_desest.loc[max_idx, 'fecha'].strftime('%b-%y')})".replace(",", "."))
print(f"  Mínimo:  {int(serie_desest.loc[min_idx, 'empleo_desest']):,} ({serie_desest.loc[min_idx, 'fecha'].strftime('%b-%y')})".replace(",", "."))
print(f"  Último:  {int(serie_desest.loc[ult_idx, 'empleo_desest']):,} ({serie_desest.loc[ult_idx, 'fecha'].strftime('%b-%y')})".replace(",", "."))

Mes          Original    Desest.     Factor
---------------------------------------------
Apr-23         34.866     33.076      1.054
May-23         34.329     33.262      1.032
Jun-23         33.740     34.630      0.974
Jul-23         32.593     34.248      0.952
Aug-23         31.886     33.815      0.943
Sep-23         31.274     33.182      0.942
Oct-23         31.051     32.989      0.941
Nov-23         30.693     32.845      0.934
Dec-23         29.977     32.360      0.926
Jan-24         31.289     32.394      0.966
Feb-24         36.709     31.726      1.157
Mar-24         33.582     30.649      1.096
Apr-24         31.589     29.967      1.054
May-24         30.678     29.725      1.032
Jun-24         28.830     29.590      0.974
Jul-24         28.005     29.427      0.952
Aug-24         27.917     29.606      0.943
Sep-24         28.020     29.730      0.942
Oct-24         27.873     29.612      0.941
Nov-24         27.826     29.777      0.934
Dec-24         27.519     29.7

In [6]:
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd()

# Archivos que podrían tener la serie mensual anterior
archivos = [
    "data/raw/SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx",
    "data/raw/provinciales_serie_empleo_trimestral_2dig_6.xlsx",
    "data/raw/provinciales_serie_empleo_anual_3.xlsx",
    "data/raw/provinciales_serie_empleo_trimestral_4dig_7.xlsx",
]

print("=" * 70)
print("BÚSQUEDA DE EMPLEO FORMAL LA RIOJA — Ene-22 a Mar-23")
print("=" * 70)
print()

# 1. SerieWeb completo — ver TODA la fila de La Rioja
RUTA = RAIZ / "data/raw/SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(RUTA, sheet_name=0, header=None)
fechas = pd.to_datetime(df.iloc[0, 1:], errors="coerce")
fila_rioja = df.iloc[12, 1:]

print("1) SerieWeb — Fila 12 (La Rioja), todos los meses:")
print(f"   Primer valor no-NaN: ", end="")
for i, v in enumerate(fila_rioja):
    if pd.notna(v):
        print(f"{fechas.iloc[i].strftime('%b-%y')} = {v}")
        break
print(f"   Último valor: {fechas.iloc[-1].strftime('%b-%y')} = {fila_rioja.iloc[-1]}")
print()

# ¿Hay otras filas con La Rioja?
print("2) Buscando otras filas con 'Rioja':")
for i in range(len(df)):
    val = str(df.iloc[i, 0])
    if "Rioja" in val or "rioja" in val:
        print(f"   Fila {i}: {val}")
print()

BÚSQUEDA DE EMPLEO FORMAL LA RIOJA — Ene-22 a Mar-23

1) SerieWeb — Fila 12 (La Rioja), todos los meses:
   Primer valor no-NaN: Jan-09 = 25.573
   Último valor: Jun-26 = 28.656

2) Buscando otras filas con 'Rioja':
   Fila 12: La Rioja



In [8]:
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd()
RUTA = RAIZ / "data/raw/SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(RUTA, sheet_name=0, header=None)

fechas = pd.to_datetime(df.iloc[0, 1:], errors="coerce")
fila_rioja = df.iloc[12, 1:]

inicio = pd.Timestamp("2022-01-01")
fin = pd.Timestamp("2023-12-31")

print(f"{'Mes':<10} {'Valor':>12}")
print("-" * 25)
for f, v in zip(fechas, fila_rioja):
    if pd.notna(f) and inicio <= f <= fin:
        if pd.notna(v):
            print(f"{f.strftime('%b-%y'):<10} {v:>12.3f}")
        else:
            print(f"{f.strftime('%b-%y'):<10} {'NaN':>12}")

Mes               Valor
-------------------------
Apr-23           34.866
May-23           34.329
Jun-23           33.740
Jul-23           32.593
Aug-23           31.886
Sep-23           31.274
Oct-23           31.051
Nov-23           30.693
Dec-23           29.977
